# Разбор A. Кинетика и план эксперимента

Временные данные способны оценить k только при подходящем плане. Гипотеза: конечный выход и пара ранних точек плохо различают скорость и конечное плато. На A изучаем условную кинетику, не приписывая её опубликованной реакции ксантинов. B проектирует измерение превращений аторвастатина.

Сгенерируйте детерминированные точки при k=.02 с небольшим учебным шумом и seed=42. Оцените k через curve_fit, сравните с истинным параметром, построите остатки. Создайте обратимый контроль с плато .25 и покажите систематическое расхождение необратимой модели. Обсудите случайные и систематические источники ошибки.

In [ ]:
from pathlib import Path
import sys,json,math
from IPython.display import display
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'programm/course.json').is_file())
sys.path.insert(0,str(ROOT/'practicuum/shared'))
from course_support import DATA,compound,esol,workspace,write_json,init_dossier
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
def publish_plot(fig,name):
    fig.savefig(OUT/name,dpi=120,bbox_inches='tight')
    display(fig)
    plt.close(fig)
OUT=workspace('M09-L02','case')


Сначала проверяем входы и единицы. Словари сохраняют названия полей вместе со значениями; список сохраняет соответствие записей. Выведите промежуточный результат перед дальнейшим анализом.

In [ ]:
import numpy as np
from scipy.optimize import curve_fit
rng=np.random.default_rng(42);t=np.array([0,10,20,40,60,90,120.])
truth=np.exp(-.02*t);observed=truth+rng.normal(0,.008,len(t))
model=lambda t,k:np.exp(-k*t)
parameters,cov=curve_fit(model,t,observed,p0=[.01],bounds=(0,np.inf))
khat=float(parameters[0]);print({'k_fit_min_inverse':khat,'half_life_min':math.log(2)/khat})


Теперь сравним модель с контрольным случаем. `assert` останавливает расчёт при нарушении контракта; это защита от ошибочного продолжения, а не оценка научного понимания.

In [ ]:
plateau=.25+.75*np.exp(-.02*t)
wrong,_=curve_fit(model,t,plateau,p0=[.01],bounds=(0,np.inf))
fig,axes=plt.subplots(1,2,figsize=(9,3));axes[0].scatter(t,observed);axes[0].plot(t,model(t,khat))
axes[0].set(xlabel='Time / min',ylabel='Simulated A/A0')
axes[1].axhline(0,color='gray');axes[1].plot(t,plateau-model(t,*wrong),'o-')
axes[1].set(xlabel='Time / min',ylabel='Plateau-model residual');publish_plot(fig,'fit.png')


Сохраним именно полученный результат. Не переносите этот вывод на другую химическую форму, мишень или экспериментальные условия без отдельной проверки.

In [ ]:
result={'kind':'simulation','k_fit':khat,'parameter_covariance':cov.tolist(),'seed':42,'experiment':'NOT_RUN'}


In [ ]:
write_json(OUT/'result.json',result)
from dossier_records import append_record
receipt=append_record(init_dossier(),'M09-L02','A',result,ROOT,inputs=[])
print('Versioned evidence:',receipt)


Возврат исходного k проверяет синтетический эксперимент, не стабильность лекарства. Новые данные должны различать конкурирующие модели. В M10 потребуется разделить соединения, чтобы вообще измерять временные концентрации.

Перед переходом ответьте: какие результаты получены вычислением, какие заданы условно, и какая независимая проверка ещё отсутствует?